# 24 — Temporal Analytics: Timestamps, DatetimeIndex, & Timedeltas
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python standard `datetime`, Pandas `Timestamp`, vectorized `.dt` accessors, `DateOffset` calendars, and high-throughput `Timedelta` duration arithmetic.*

---

## 📌 Executive Summary & Interview Expectations
Time-series and event telemetry data power nearly every modern data stack (financial tick data, rideshare trips, e-commerce deliveries, clickstreams). In technical interviews, examiners focus on:
1. **Datetime Parsing & Format Safety**: Why heuristic parsing without explicit `format` is dangerous and slow, and how `errors='coerce'` prevents pipeline crashes.
2. **The `.dt` Accessor Suite**: Vectorized calendar extraction (`day_name()`, `is_quarter_start`, `dayofweek`).
3. **Calendar Offsets vs Fixed Durations**: `pd.DateOffset` / `pd.offsets.MonthEnd()` (which respect varying month lengths and leap years) vs `pd.Timedelta` (which represents fixed absolute time).
4. **The Notorious Timedelta Trap**: The catastrophic distinction between `.dt.seconds` (remainder seconds within the day) and `.dt.total_seconds()` (true total elapsed time).
5. **Real-World Case Studies**: Analyzing Disney equities, delivery logistics, and CitiBike mobility patterns.

## 1. Standard Python `datetime` vs Pandas `Timestamp`

### Architectural Difference:
- Python's `datetime.datetime` is a standard object allocated individually on the heap.
- Pandas `pd.Timestamp` is a subclass of Python's `datetime.datetime` backed by NumPy's 64-bit integer nanosecond representation (`datetime64[ns]`).

In [1]:
import datetime as dt
import numpy as np
import pandas as pd

# Python standard library datetime objects
birthday = dt.date(1991, 4, 12)
alarm_clock = dt.time(6, 43, 25)
moon_landing = dt.datetime(1969, 7, 20, 22, 56, 15)

print("Python Date:", birthday, "| Year:", birthday.year, "| Month:", birthday.month)
print("Python Time:", alarm_clock, "| Hour:", alarm_clock.hour, "| Minute:", alarm_clock.minute)
print("Python DateTime:", moon_landing)

# Pandas Timestamp (compatible with Python datetime)
ts = pd.Timestamp("1991-04-12 06:43:25")
print("\nPandas Timestamp:", ts)
print("Equality check with dt.datetime:", ts == dt.datetime(1991, 4, 12, 6, 43, 25))

Python Date: 1991-04-12 | Year: 1991 | Month: 4
Python Time: 06:43:25 | Hour: 6 | Minute: 43
Python DateTime: 1969-07-20 22:56:15

Pandas Timestamp: 1991-04-12 06:43:25
Equality check with dt.datetime: True


## 2. High-Performance Ingestion: `pd.to_datetime`

### 💡 Interview Tip: Explicit `format` and `errors='coerce'`
- If date strings follow an ISO or fixed format (e.g. `YYYY-MM-DD`), pass `format='%Y-%m-%d'`. This is **~10x-50x faster** than letting Pandas guess the format!
- Use `errors='coerce'` to convert corrupt date strings into `NaT` (Not-a-Time) instead of crashing the pipeline.

In [2]:
# Batch string conversion to DatetimeIndex
date_strings = ["2023-01-15", "2023-02-28", "2023-03-31", "invalid_date"]
parsed_dates = pd.to_datetime(date_strings, errors="coerce")

print("Parsed DatetimeIndex with NaT handling:")
print(parsed_dates)

Parsed DatetimeIndex with NaT handling:
DatetimeIndex(['2023-01-15', '2023-02-28', '2023-03-31', 'NaT'], dtype='datetime64[us]', freq=None)


## 3. Case Study 1: Disney Equities & The `.dt` Accessor Suite

In [3]:
# Ingest Disney stock history
disney = pd.read_csv("disney.csv")
disney["Date"] = pd.to_datetime(disney["Date"])

print(f"Disney Data Shape: {disney.shape} | Date dtype: {disney['Date'].dtype}")
disney.head(3)

Disney Data Shape: (14727, 5) | Date dtype: datetime64[us]


,Date,High,Low,Open,Close
0,1962-01-02,0.096026,0.092908,0.092908,0.092908
1,1962-01-03,0.094467,0.092908,0.092908,0.094155
2,1962-01-04,0.094467,0.093532,0.094155,0.094155


In [4]:
# Extract calendar features via .dt accessor
disney["Day_of_Week"] = disney["Date"].dt.day_name()
disney["Year"] = disney["Date"].dt.year
disney["Is_Quarter_End"] = disney["Date"].dt.is_quarter_end

print("Day of Week Trading Close Price Analysis:")
display(disney.groupby("Day_of_Week")["Close"].mean().round(2))

print("\nSample Trading Days that fell on Quarter-End:")
display(disney[disney["Is_Quarter_End"]].head(3))

Day of Week Trading Close Price Analysis:


Day_of_Week
Friday       23.55
Monday       23.16
Thursday     23.54
Tuesday      23.56
Wednesday    23.61
Name: Close, dtype: float64


Sample Trading Days that fell on Quarter-End:


,Date,High,Low,Open,Close,Day_of_Week,Year,Is_Quarter_End
251,1962-12-31,0.074501,0.071290,0.074501,0.072253,Monday,1962,True
440,1963-09-30,0.109825,0.105972,0.108541,0.107577,Monday,1963,True
502,1963-12-31,0.101476,0.096980,0.097622,0.101476,Tuesday,1963,True


## 4. Calendar Math: `pd.DateOffset` vs Business Offsets

### 💡 `DateOffset` vs `Timedelta`
- `pd.Timedelta(days=30)` adds exactly $30 \times 86400$ seconds.
- `pd.DateOffset(months=1)` shifts the date by **1 calendar month**, properly adjusting for February (28/29 days) and April (30 days)!
- Business offsets like `pd.offsets.BMonthEnd()` snap directly to the final business day of the month.

In [5]:
sample_date = pd.Timestamp("2024-01-31")

print("Original Date:", sample_date)
print("Plus 1 Calendar Month (DateOffset):", sample_date + pd.DateOffset(months=1))
print("Plus MonthEnd Offset:", sample_date + pd.offsets.MonthEnd(1))
print("Plus Business MonthEnd Offset (BMonthEnd):", sample_date + pd.offsets.BMonthEnd(1))

Original Date: 2024-01-31 00:00:00
Plus 1 Calendar Month (DateOffset): 2024-02-29 00:00:00
Plus MonthEnd Offset: 2024-02-29 00:00:00
Plus Business MonthEnd Offset (BMonthEnd): 2024-02-29 00:00:00


## 5. Case Study 2: Delivery Logistics & The Timedelta Trap

### 🚨 Top Interview Gotcha: `.dt.seconds` vs `.dt.total_seconds()`
- `duration.dt.days`: Returns the integer number of full days.
- `duration.dt.seconds`: Returns the **remainder seconds within the final day** ($0$ to $86,399$), NOT total seconds!
- `duration.dt.total_seconds()`: Returns the **true total elapsed duration** in fractional seconds!
- Using `.dt.seconds / 3600` on a 5-day delivery returns $< 24$ hours, corrupting shipping metrics!

In [6]:
deliveries = pd.read_csv("deliveries.csv")
deliveries["order_date"] = pd.to_datetime(deliveries["order_date"])
deliveries["delivery_date"] = pd.to_datetime(deliveries["delivery_date"])

# Vectorized duration calculation
deliveries["duration"] = deliveries["delivery_date"] - deliveries["order_date"]

# Demonstrating the trap
sample_row = deliveries.iloc[0]["duration"]
print(f"Sample Duration: {sample_row}")
print(f"  -> .days: {sample_row.days}")
print(f"  -> .seconds (REMAINDER ONLY!): {sample_row.seconds}")
print(f"  -> .total_seconds() (TRUE DURATION): {sample_row.total_seconds():,.0f} seconds")
print(f"  -> True Duration in Hours: {sample_row.total_seconds() / 3600:.2f} hours")

print("\nDelivery Summary Statistics:")
display(deliveries["duration"].describe())

Sample Duration: 257 days 00:00:00
  -> .days: 257
  -> .seconds (REMAINDER ONLY!): 0
  -> .total_seconds() (TRUE DURATION): 22,204,800 seconds
  -> True Duration in Hours: 6168.00 hours

Delivery Summary Statistics:


/var/folders/54/5j97z6452x19yddj6yv2l7j00000gn/T/ipykernel_27306/857834182.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  deliveries["order_date"] = pd.to_datetime(deliveries["order_date"])
/var/folders/54/5j97z6452x19yddj6yv2l7j00000gn/T/ipykernel_27306/857834182.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  deliveries["delivery_date"] = pd.to_datetime(deliveries["delivery_date"])


count                          501
mean     1217 days 22:53:53.532934
std       917 days 21:51:05.029340
min                8 days 00:00:00
25%              423 days 00:00:00
50%              998 days 00:00:00
75%             1901 days 00:00:00
max             3583 days 00:00:00
Name: duration, dtype: object

## 6. Case Study 3: CitiBike Mobility Analytics

### Weekly Normalization via Timedelta Math
To aggregate high-frequency trip timestamps by week:
Subtract `start_time.dt.dayofweek` days to normalize every timestamp down to Monday 00:00:00!

In [7]:
citi_bike = pd.read_csv("citibike.csv")
citi_bike["start_time"] = pd.to_datetime(citi_bike["start_time"])
citi_bike["stop_time"] = pd.to_datetime(citi_bike["stop_time"])

# Compute trip duration
citi_bike["duration"] = citi_bike["stop_time"] - citi_bike["start_time"]

# Round down to Monday of the trip week
days_from_monday = pd.to_timedelta(citi_bike["start_time"].dt.dayofweek, unit="day")
citi_bike["week_starting_monday"] = (citi_bike["start_time"] - days_from_monday).dt.date

print("Trips by Week Starting Monday:")
display(citi_bike["week_starting_monday"].value_counts().sort_index())

print("\nTop 3 Longest Duration Trips (Outlier Inspection):")
display(citi_bike.nlargest(3, columns="duration"))

Trips by Week Starting Monday:


week_starting_monday
2020-06-01    11637
2020-06-08    11671
2020-06-15    11622
2020-06-22    11710
2020-06-29     3360
Name: count, dtype: int64


Top 3 Longest Duration Trips (Outlier Inspection):


,start_time,stop_time,duration,week_starting_monday
0,2020-06-26 16:25:10,2020-07-29 07:25:10.000,32 days 15:00:00,2020-06-22
1,2020-06-26 19:11:24,2020-07-27 20:11:24.000,31 days 01:00:00,2020-06-22
48969,2020-06-19 09:43:27,2020-06-19 12:41:34.233,0 days 02:58:07.233000,2020-06-15


---
## 🎯 7. Technical Interview Corner: Tricky Questions & Drills

### Q1: What is the difference between Timezone Localization and Timezone Conversion?
**Answer**:
1. **`tz_localize('UTC')`**:
   - Takes a **naive** datetime (with no timezone attached) and labels it with a timezone without altering the clock time.
   - `Timestamp('2024-01-01 12:00').tz_localize('UTC')` $\rightarrow$ `2024-01-01 12:00:00+00:00`.
2. **`tz_convert('America/New_York')`**:
   - Takes an **already timezone-aware** datetime and converts it to a different timezone, shifting the clock time according to UTC offset and daylight saving time.
   - `Timestamp('2024-01-01 12:00:00+00:00').tz_convert('America/New_York')` $\rightarrow$ `2024-01-01 07:00:00-05:00`.

---

### Q2: Advanced Interview Coding Challenge: Fast SLA Business-Hour Calculation
**Challenge**:
Logistics SLA states that orders must be delivered within **48 calendar hours**.
1. Flag each delivery in `deliveries` as `'SLA Breach'` or `'SLA Met'`.
2. Calculate the breach rate (%) across the entire fleet!

In [8]:
# Interview Solution: High-Throughput Timedelta SLA Auditing
sla_threshold = pd.Timedelta(hours=48)

deliveries_audited = deliveries.assign(
    sla_status=np.where(deliveries["duration"] > sla_threshold, "SLA Breach", "SLA Met"),
    turnaround_hours=(deliveries["duration"].dt.total_seconds() / 3600).round(1)
)

breach_rate = (deliveries_audited["sla_status"] == "SLA Breach").mean() * 100
print(f"Fleet SLA Breach Rate: {breach_rate:.2f}%")

print("\nSample Audited Deliveries:")
display(deliveries_audited[["order_date", "delivery_date", "turnaround_hours", "sla_status"]].head(5))

Fleet SLA Breach Rate: 100.00%

Sample Audited Deliveries:


,order_date,delivery_date,turnaround_hours,sla_status
0,1998-05-24,1999-02-05,6168.0,SLA Breach
1,1992-04-22,1998-03-06,51456.0,SLA Breach
2,1991-02-10,1992-08-26,13512.0,SLA Breach
3,1992-07-21,1997-11-20,46752.0,SLA Breach
4,1993-09-02,1998-06-10,41808.0,SLA Breach
